# RBC box detector -- yolo11s on the expanded dataset

Same single-class box detector as `yolo11n_1024_singlecls`, but:
- **yolo11s** (small) instead of nano -- more capacity, should help recall on the ~1.5% of cells the nano model barely registers (~0.002-0.003 confidence).
- **3500 train / 500 val images** instead of 330 -- converted from `yolo_seg_dataset`'s Cellpose-bootstrapped polygon masks (a box is just the min/max x,y of a polygon, so no new manual labeling was needed).

Run cells top to bottom. Training logs print live below the train cell as it runs.

In [1]:
from pathlib import Path
from ultralytics import YOLO

PROJECT_DIR  = Path('F:/Livo/Data - 2026/Rbc/rbc_yolo')
DATASET_YAML = 'F:/Livo/Data - 2026/Rbc/yolo_dataset_expanded/dataset.yaml'

IMGSZ    = 1024
BATCH    = 4
EPOCHS   = 50
PATIENCE = 30
BASE_MODEL = 'yolo11s.pt'
RUN_NAME   = f'{Path(BASE_MODEL).stem}_{IMGSZ}_singlecls_expanded'

print(Path(DATASET_YAML).read_text())

path: F:/Livo/Data - 2026/Rbc/yolo_dataset_expanded
train: images/train
val: images/val
nc: 1
names:
  0: rbc



## Train

**Only run this if nothing else is training** -- the 8GB card fits one job at a time. If you get a background-training notification / see another `python.exe` process already using the GPU, stop that first (or ask to have it stopped).

In [2]:
model = YOLO(BASE_MODEL)
result = model.train(
    data           = DATASET_YAML,
    epochs         = EPOCHS,
    patience       = PATIENCE,
    imgsz          = IMGSZ,
    batch          = BATCH,
    project        = str(PROJECT_DIR),
    name           = RUN_NAME,
    device         = 0,
    workers        = 4,
    optimizer      = 'AdamW',
    lr0            = 0.001,
    lrf            = 0.01,
    cos_lr         = True,
    warmup_epochs  = 5,
    single_cls     = True,
    hsv_h          = 0.0,
    hsv_s          = 0.0,
    hsv_v          = 0.02,
    fliplr         = 0.5,
    flipud         = 0.5,
    degrees        = 15.0,
    scale          = 0.3,
    translate      = 0.1,
    mosaic         = 0.0,
    mixup          = 0.0,
    copy_paste     = 0.0,
    erasing        = 0.0,
    box            = 7.5,
    cls            = 0.5,
    dfl            = 1.5,
    save           = True,
    save_period    = 10,
    val            = True,
    plots          = True,
)

if hasattr(result, 'results_dict'):
    m = result.results_dict
    print('mAP50    :', m.get('metrics/mAP50(B)', 'n/a'))
    print('mAP50-95 :', m.get('metrics/mAP50-95(B)', 'n/a'))
    print('Precision:', m.get('metrics/precision(B)', 'n/a'))
    print('Recall   :', m.get('metrics/recall(B)', 'n/a'))

New https://pypi.org/project/ultralytics/8.4.137 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.67  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=F:/Livo/Data - 2026/Rbc/yolo_dataset_expanded/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.02, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.93

## Check progress / logs without waiting on the train cell

Run this cell any time (from a second notebook, or after interrupting) to see the latest completed epochs without blocking on the training cell above.

In [ ]:
import pandas as pd

results_csv = PROJECT_DIR / RUN_NAME / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print(f'{len(df)} epochs completed so far')
    display(df.tail(10))
else:
    print(f'no results.csv yet at {results_csv}')

## Validate the best checkpoint

In [ ]:
best_weights = PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt'
best_model = YOLO(str(best_weights))
metrics = best_model.val(data=DATASET_YAML)

print('mAP50    :', metrics.box.map50)
print('mAP50-95 :', metrics.box.map)
print('Precision:', metrics.box.mp)
print('Recall   :', metrics.box.mr)